# Q2 actions: `KEEP_POOLED`, `RECALIBRATE`, `SPLIT`

Q1 asks whether the pooled logistic score is good enough on a segment.
Q2 asks what to do when it is not:

| Action | When |
| --- | --- |
| `KEEP_POOLED` | Rank order and calibration are fine, or a refit would not add enough. |
| `RECALIBRATE` | Rank order is fine, but the PD level is off (O/E, ECE). |
| `SPLIT` | Rank order (or shape) is broken, the segment is important, a same-predictor refit beats the pooled score on a forward holdout, and the predictors are stable over vintages. |
| `NONE` | Too few rows / events to decide (`INCONCLUSIVE`). |

`make_synthetic_book` builds four channels that hit each of those:

* **core** — pooled score is good → `KEEP_POOLED`
* **miscal** — ranking holds, PDs are inflated → `RECALIBRATE`
* **inverted** — `x1` sign is reversed vs the pooled model → `SPLIT`
* **tiny** — too small → `NONE`

This notebook uses the same book and gates as `tests/test_evaluate.py`, so the
printed actions match the tests. The package targets Python 3.6+, so nothing
here uses 3.7+ syntax.

In [1]:
import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from scorecard_segment_eval import (
    BinningModel,
    Gates,
    action_list,
    decision_table,
    evaluate_segments,
    make_synthetic_book,
    parse_scorecard_sql_path,
)
from scorecard_segment_eval.metrics import ece, gini, observed_expected

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 80)


def _find_repo_root():
    here = os.path.abspath(os.getcwd())
    for candidate in (here, os.path.dirname(here)):
        if os.path.isdir(os.path.join(candidate, "scorecard_segment_eval")):
            return candidate
    return here


REPO_ROOT = _find_repo_root()
WORKDIR = os.path.join(REPO_ROOT, "notebooks", "q2_output")
if not os.path.isdir(WORKDIR):
    os.makedirs(WORKDIR)

df, cols = make_synthetic_book(n=9000, seed=3)
gates = Gates(
    min_n=400,
    min_events=25,
    n_bootstrap=80,
    bootstrap_seed=0,
    holdout_frac=0.3,
)
print("rows", len(df), "channels", sorted(df["channel"].unique().tolist()))
print("artefacts ->", WORKDIR)

rows 9000 channels ['core', 'inverted', 'miscal', 'tiny']
artefacts -> /workspace/notebooks/q2_output


## 1. Run the evaluation

Fit a portfolio grouping on the observable book (so PSI and grouping comparison
use the same bins), then call `evaluate_segments`. Each Q2 action below is
read off `result.decisions` — nothing is hard-coded.

In [2]:
obs = df.loc[df[cols.col_obs].eq(1)]
grouping = BinningModel.fit(obs[list(cols.cols_pred)], obs[cols.col_target], gates)
print(grouping.iv_table())

result = evaluate_segments(df, cols, gates, grouping=grouping)
print(decision_table(result))
print()
print(action_list(result))

  feature         kind           method  n_bins        iv  monotonic                          notes
0      x1      numeric             tree       5  0.150374       True  merged_monotonicity_violation
1      x2      numeric             tree       6  0.192471       True                               
2     cat  categorical  categorical_woe       3  0.024120      False                               


  segment_col segment_value  important    q1_verdict                    failed_pillars    q2_action                       q2_reason      gini  gini_ratio        oe       ece  volume_share  default_share    ar_gap  \
0     channel          core       True          GOOD                                    KEEP_POOLED                performance_good  0.561509    2.258336  1.145405  0.024745      0.591111       0.555851  0.049150   
1     channel      inverted       True          WEAK  rank_order,calibration,stability        SPLIT    holdout_delta_gini_and_shape -0.521175   -2.096119  1.622751  0.250424      0.220000       0.275266  0.064937   
2     channel        miscal       True          WEAK                       calibration  RECALIBRATE  rank_order_ok_calibration_fail  0.546015    2.196021  0.491198  0.179955      0.180000       0.163564 -0.244742   
3     channel          tiny      False  INCONCLUSIVE                insufficient_power         NONE              insufficient_power  0.5

In [3]:
dec = result.decisions.set_index("segment_value")
ref = result.refit_comparison.set_index("segment_value")
print("Q2 actions:")
print(dec[["q1_verdict", "failed_pillars", "q2_action", "q2_reason"]].to_string())
assert dec.loc["core", "q2_action"] == "KEEP_POOLED"
assert dec.loc["miscal", "q2_action"] == "RECALIBRATE"
assert dec.loc["inverted", "q2_action"] == "SPLIT"
assert dec.loc["tiny", "q2_action"] == "NONE"
print("\nassertions passed")

Q2 actions:
                 q1_verdict                    failed_pillars    q2_action                       q2_reason
segment_value                                                                                             
core                   GOOD                                    KEEP_POOLED                performance_good
inverted               WEAK  rank_order,calibration,stability        SPLIT    holdout_delta_gini_and_shape
miscal                 WEAK                       calibration  RECALIBRATE  rank_order_ok_calibration_fail
tiny           INCONCLUSIVE                insufficient_power         NONE              insufficient_power

assertions passed


## 2. `KEEP_POOLED` — channel `core`

The pooled score already ranks and calibrates. Q1 is `GOOD`, so Q2 does not
look for a split. A same-predictor refit is still fitted for the comparison
table; it is not adopted.

In [4]:
core = dec.loc["core"]
print("q1_verdict     ", core["q1_verdict"])
print("failed_pillars ", core["failed_pillars"])
print("q2_action      ", core["q2_action"])
print("q2_reason      ", core["q2_reason"])
print()
show = [
    "gini", "gini_ratio", "oe", "ece",
]
print(core[[c for c in show if c in core.index]].to_string())
print()
print(result.segment_summary.loc[
    result.segment_summary["segment_value"].eq("core")
    & result.segment_summary["slice"].eq("observable"),
    ["n", "defaults", "gini", "oe", "ece"],
].to_string(index=False))

q1_verdict      GOOD
failed_pillars  
q2_action       KEEP_POOLED
q2_reason       performance_good

gini          0.561509
gini_ratio    2.258336
oe            1.145405
ece           0.024745

     n  defaults     gini       oe      ece
4649.0     836.0 0.561509 1.145405 0.024745


## 3. `RECALIBRATE` — channel `miscal`

The synthetic book inflates PDs on `miscal` (`pd * 2.4`) without flipping
rank order. Q1 fails **calibration** only, so Q2 is `RECALIBRATE`:
`logit(p*) = a + b * logit(p)` on the training slice, applied to holdout.

Gini stays (ranking is unchanged); Brier / log-loss and O/E should improve.

In [5]:
miscal = dec.loc["miscal"]
print("q1_verdict     ", miscal["q1_verdict"])
print("failed_pillars ", miscal["failed_pillars"])
print("q2_action      ", miscal["q2_action"])
print("q2_reason      ", miscal["q2_reason"])
print()
print("gini (ranking should be fine)")
print("  pooled     ", miscal["gini"])
print("  gini_ratio ", miscal["gini_ratio"])
print("calibration (this is what fails)")
print("  oe          ", miscal["oe"])
print("  ece         ", miscal["ece"])
print("holdout after intercept/slope rescale")
print(ref.loc["miscal", ["gini_pooled", "gini_recal", "brier_pooled", "brier_recal"]].to_string())

seg = obs.loc[obs["channel"].eq("miscal")]
oe = observed_expected(seg[cols.col_target], seg[cols.col_score])
print("\nobservable slice O/E", oe["oe"], "obs_rate", oe["obs_rate"], "mean_pd", oe["mean_pd"])
print("observable slice ECE", ece(seg[cols.col_target], seg[cols.col_score]))

q1_verdict      WEAK
failed_pillars  calibration
q2_action       RECALIBRATE
q2_reason       rank_order_ok_calibration_fail

gini (ranking should be fine)
  pooled      0.5460148704051144
  gini_ratio  2.1960205212811283
calibration (this is what fails)
  oe           0.4911983483370655
  ece          0.17995481373192349
holdout after intercept/slope rescale
gini_pooled     0.538516
gini_recal      0.538516
brier_pooled    0.184446
brier_recal     0.126098

observable slice O/E 0.4911983483370655 obs_rate 0.17372881355932204 mean_pd 0.3536836272912455
observable slice ECE 0.17995481373192349


## 4. `SPLIT` — channel `inverted`

`x1` is a *negative* risk driver on this channel and a *positive* one on the
rest of the book. The pooled score therefore ranks inverted applications the
wrong way. Q1 fails `rank_order`. A same-predictor refit (new WoE bins + new
coefficients, same features) recovers Gini on a forward holdout, the predictors
are stable, and Q2 is `SPLIT`.

The grouping comparison against the portfolio bins is the smoking gun: WoE
sign flips on `x1`.

In [6]:
inv = dec.loc["inverted"]
print("q1_verdict     ", inv["q1_verdict"])
print("failed_pillars ", inv["failed_pillars"])
print("q2_action      ", inv["q2_action"])
print("q2_reason      ", inv["q2_reason"])
print("important      ", inv["important"])
print()
print("holdout discrimination")
print(ref.loc["inverted", ["gini_pooled", "gini_refit", "delta_gini", "delta_gini_ci_low"]].to_string())
print()
print("stability")
print("  pass ", inv["stability_pass"], " score", inv["stability_score"])

cmp_inv = result.grouping_comparison
if not cmp_inv.empty and "segment_value" in cmp_inv.columns:
    cmp_inv = cmp_inv.loc[cmp_inv["segment_value"].astype(str).eq("inverted")]
print("\ngrouping comparison (inverted vs portfolio), x1 first")
if cmp_inv.empty:
    print("(no grouping_comparison rows)")
else:
    x1_cmp = cmp_inv.loc[cmp_inv["feature"].eq("x1")]
    cols_show = [c for c in ["feature", "segment_bin", "portfolio_bins", "woe_segment", "woe_portfolio", "woe_delta", "kind", "note"] if c in cmp_inv.columns]
    print((x1_cmp if not x1_cmp.empty else cmp_inv).head(12)[cols_show].to_string(index=False))

q1_verdict      WEAK
failed_pillars  rank_order,calibration,stability
q2_action       SPLIT
q2_reason       holdout_delta_gini_and_shape
important       True

holdout discrimination
gini_pooled         -0.497277
gini_refit           0.616694
delta_gini           1.113972
delta_gini_ci_low    0.986092

stability
  pass  True  score 0.7735429446980318

grouping comparison (inverted vs portfolio), x1 first
feature                                                                                                              segment_bin                                                                              portfolio_bins  woe_segment  woe_portfolio  woe_delta                     kind                                                                                                                                                                                      note
     x1                                                                                                         (-inf, -1.

### WoE grouping vs vintage

`BinningModel.plot_vintage_stability` draws three stacked subplots for one
predictor: **true event rate**, **bin share**, and **univariate Gini**. Use it
to see whether a grouping that looks fine in pool is drifting month to month.

On `inverted`, `x1` should keep a strong univariate Gini (the signal is real)
while the event-rate order of the bins is the *opposite* of the portfolio.

In [7]:
inv_obs = obs.loc[obs["channel"].eq("inverted")].copy()
seg_grouping = BinningModel.fit(
    inv_obs[list(cols.cols_pred)], inv_obs[cols.col_target], gates
)
table = seg_grouping.vintage_stability_table(
    inv_obs, inv_obs[cols.col_target], cols.col_date, "x1", min_rows=40
)
print("vintage table head")
print(table.head(12).to_string(index=False))
print("\nvintages", table["vintage"].nunique(), "median univariate Gini", table.drop_duplicates("vintage")["univariate_gini"].median())

fig, axes = seg_grouping.plot_vintage_stability(
    inv_obs, inv_obs[cols.col_target], cols.col_date, "x1", min_rows=40
)
plot_path = os.path.join(WORKDIR, "inverted_x1_vintage_stability.png")
fig.savefig(plot_path, dpi=120)
plt.close(fig)
print("wrote", plot_path)

vintage table head
feature vintage                    bin    n  events    share  event_rate  univariate_gini
     x1 2023-01       (-inf, -1.66933]  6.0     5.0 0.055556    0.833333         0.665962
     x1 2023-01   (-1.66933, -1.06844]  9.0     7.0 0.083333    0.777778         0.665962
     x1 2023-01  (-1.06844, -0.660312] 16.0     4.0 0.148148    0.250000         0.665962
     x1 2023-01 (-0.660312, -0.117337] 12.0     2.0 0.111111    0.166667         0.665962
     x1 2023-01 (-0.117337, 0.0895581] 10.0     1.0 0.092593    0.100000         0.665962
     x1 2023-01       (0.0895581, inf) 55.0     3.0 0.509259    0.054545         0.665962
     x1 2023-02       (-inf, -1.66933]  2.0     2.0 0.021505    1.000000         0.529161
     x1 2023-02   (-1.66933, -1.06844] 10.0     4.0 0.107527    0.400000         0.529161
     x1 2023-02  (-1.06844, -0.660312]  7.0     4.0 0.075269    0.571429         0.529161
     x1 2023-02 (-0.660312, -0.117337] 21.0     5.0 0.225806    0.238095         

wrote /workspace/notebooks/q2_output/inverted_x1_vintage_stability.png


## 5. `NONE` — channel `tiny`

Eighty rows is below `Gates.min_n`. Q1 is `INCONCLUSIVE` and Q2 is `NONE`:
there is not enough power to choose a treatment.

In [8]:
tiny = dec.loc["tiny"]
print("q1_verdict ", tiny["q1_verdict"])
print("q2_action  ", tiny["q2_action"])
print("q2_reason  ", tiny["q2_reason"])
tiny_sum = result.segment_summary.loc[
    result.segment_summary["segment_value"].eq("tiny")
    & result.segment_summary["slice"].eq("observable")
]
print(tiny_sum[["n", "defaults"]].to_string(index=False))

q1_verdict  INCONCLUSIVE
q2_action   NONE
q2_reason   insufficient_power
   n  defaults
69.0       8.0


## 6. Logit form (`_VAL` / `_LIN`) on a SQL scorecard

When the production model uses `nvl(LN(p/(1-p)), impute) as featE_VAL`, a
refit must keep that predictor as `log(p/(1-p))` rather than re-binning it as
WoE. `ScorecardColumns.logit_pred_cols` prefers a VAL/LIN alias over a WoE
alias. `evaluate_segments` passes that list into the segment refit.

In [9]:
sql_path = os.path.join(REPO_ROOT, "tests", "fixtures", "sample_scorecard.sql")
parsed = parse_scorecard_sql_path(sql_path)
sql_cols = parsed.to_columns(col_id="id")
print("pred_map (excerpt)")
for raw in ("indosat_v2", "featureB", "featE", "featF_v3_0"):
    print(" ", raw, "->", sql_cols.pred_map.get(raw))
print("WoE map ", sql_cols.pred_woe_map())
print("VAL map ", sql_cols.pred_val_map())
print("logit_pred_cols", sql_cols.logit_pred_cols(grouping=parsed.grouping))
print()
print(parsed.grouping.iv_table()[["feature", "kind", "method", "notes"]].to_string(index=False))

pred_map (excerpt)
  indosat_v2 -> feature_a_WOE
  featureB -> featureB_WOE
  featE -> featE_VAL
  featF_v3_0 -> featF_v3_0_VAL
WoE map  {'indosat_v2': 'feature_a_WOE', 'featureB': 'featureB_WOE', 'featC': 'featC_WOE', 'feat_pred_D': 'feat_pred_D_WOE'}
VAL map  {'featE': 'featE_VAL', 'featF_v3_0': 'featF_v3_0_VAL', 'featG_V2': 'featG_V2_VAL', 'featH_v4_0': 'featH_v4_0_VAL', 'feati_v3': 'feati_v3_VAL', 'var_v2': 'var_v2_VAL'}
logit_pred_cols ['featE', 'featF_v3_0', 'featG_V2', 'featH_v4_0', 'feati_v3', 'var_v2']

    feature        kind    method                              notes
 indosat_v2     numeric  sql_case null_impute: -0.008113478679658392
   featureB       mixed  sql_case  null_impute: -0.19802888585164036
      featC     numeric  sql_case   null_impute: 0.04965688662575474
feat_pred_D categorical  sql_case   null_impute: 0.09577533753541978
      featE       logit sql_logit    null_impute: -2.701124677318522
 featF_v3_0       logit sql_logit                 no_null_imputation